In [3]:
import xml.etree.ElementTree as ET

xml_file = r"C:\Users\Diego Castañeda\Documents\PACS-IA Assist\src\data\LIDC-IDRI_subset\manifest-1756535103984\LIDC-IDRI\LIDC-IDRI-0001\01-01-2000-NA-NA-30178\3000566.000000-NA-03192\069.xml"

tree = ET.parse(xml_file)
root = tree.getroot()

for session in root.findall(".//readingSession"):
    reader_id = session.findtext("readerID")
    print(f"Radiólogo: {reader_id}")
    for nodule in session.findall("unblindedReadNodule"):
        nodule_id = nodule.findtext("noduleID")
        print(f"  Nódulo ID: {nodule_id}")
        for roi in nodule.findall("roi"):
            z = roi.findtext("imageZposition")
            coords = [(int(em.findtext("xCoord")), int(em.findtext("yCoord"))) 
                      for em in roi.findall("edgeMap")]
            print(f"    Slice Z={z}, {len(coords)} puntos en contorno")


In [5]:
import os
import glob
import numpy as np
import pydicom
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

# === CONFIGURA TUS RUTAS AQUÍ ===
DICOM_DIR = r"C:\Users\Diego Castañeda\Documents\PACS-IA Assist\src\data\LIDC-IDRI_subset\manifest-1756535103984\LIDC-IDRI\LIDC-IDRI-0001\01-01-2000-NA-NA-30178\3000566.000000-NA-03192"
XML_FILE  = r"C:\Users\Diego Castañeda\Documents\PACS-IA Assist\src\data\LIDC-IDRI_subset\manifest-1756535103984\LIDC-IDRI\LIDC-IDRI-0001\01-01-2000-NA-NA-30178\3000566.000000-NA-03192\069.xml"

# Si quieres filtrar qué nódulo mostrar (por id del XML); si lo dejas en None, toma el primero
NODULE_ID_FILTER = None  # por ejemplo "Nodule 1"

# === UTILIDADES ===
def load_dicom_stack(dicom_dir):
    """Carga todos los DICOM de un estudio. Devuelve dicts indexados por SOPInstanceUID y por Z."""
    dicom_files = sorted(glob.glob(os.path.join(dicom_dir, "*.dcm")))
    by_sop = {}
    by_z   = []
    for fp in dicom_files:
        ds = pydicom.dcmread(fp)
        arr = ds.pixel_array.astype(np.float32)

        # Normalización simple para visualización
        arr = (arr - arr.min()) / max(1e-6, (arr.max() - arr.min()))

        sop = getattr(ds, "SOPInstanceUID", None)
        ipp = getattr(ds, "ImagePositionPatient", None)
        z = float(ipp[2]) if ipp is not None else None

        by_sop[sop] = {"arr": arr, "ds": ds, "z": z, "path": fp}
        if z is not None:
            by_z.append((z, sop))
    # ordenar por z por si quieres navegar
    by_z.sort(key=lambda t: t[0])
    return by_sop, by_z

def parse_lidc_xml(xml_path):
    """Parsea el XML de LIDC y devuelve una lista de nódulos con sus ROIs y metadatos."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    nodules = []
    for session in root.findall(".//readingSession"):
        reader = session.findtext("readerID")
        for n in session.findall("unblindedReadNodule"):
            nid = n.findtext("noduleID")
            rois = []
            for roi in n.findall("roi"):
                z = roi.findtext("imageZposition")
                sop = roi.findtext("imageSOP_UID")  # algunos XML lo traen
                edge = []
                for em in roi.findall("edgeMap"):
                    # LIDC usa coords 1-based con origen en esquina superior izquierda
                    x = int(em.findtext("xCoord")) - 1
                    y = int(em.findtext("yCoord")) - 1
                    edge.append((x, y))
                rois.append({
                    "z": float(z) if z is not None else None,
                    "sop": sop,
                    "edge": edge
                })
            nodules.append({
                "reader": reader,
                "nodule_id": nid,
                "rois": rois
            })
    return nodules

def find_slice(by_sop, by_z, roi, z_tol=1e-3):
    """Encuentra el slice DICOM para un ROI por SOPInstanceUID (preferido) o por Z con tolerancia."""
    if roi["sop"] and roi["sop"] in by_sop:
        return by_sop[roi["sop"]]
    # fallback: por Z
    target_z = roi["z"]
    if target_z is None:
        return None
    # buscar el z más cercano
    best = None
    best_d = 1e9
    for z, sop in by_z:
        d = abs(z - target_z)
        if d < best_d:
            best_d = d
            best = by_sop[sop]
    # opcional: exigir que la diferencia sea pequeña
    if best_d > 0.5:  # mm; ajusta si tus series tienen spacing distinto
        # aún así lo devolvemos, pero te avisamos
        print(f"[WARN] ROI Z={target_z:.3f} emparejado a slice Z={best['z']:.3f} (Δ={best_d:.3f})")
    return best

def plot_roi_on_slice(img, edge_points, title=""):
    """Dibuja la imagen y el contorno del ROI."""
    h, w = img.shape
    # filtra puntos dentro del frame por seguridad
    pts = [(x, y) for (x, y) in edge_points if 0 <= x < w and 0 <= y < h]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]

    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    if len(pts) > 2:
        # ojo: matplotlib usa (x,y) como (columna, fila) → (xs, ys)
        plt.plot(xs + [xs[0]], ys + [ys[0]], linewidth=1.5)
    else:
        plt.scatter(xs, ys, s=8)
    plt.title(title)
    plt.axis("off")
    plt.show()

# === MAIN ===
if __name__ == "__main__":
    by_sop, by_z = load_dicom_stack(DICOM_DIR)
    nodules = parse_lidc_xml(XML_FILE)

    if not nodules:
        raise SystemExit("No se encontraron nódulos en el XML.")

    # Escoge nódulo (por filtro o el primero)
    nodule = None
    if NODULE_ID_FILTER:
        for n in nodules:
            if n["nodule_id"] == NODULE_ID_FILTER:
                nodule = n
                break
    if nodule is None:
        nodule = nodules[0]

    print(f"Nódulo seleccionado: {nodule['nodule_id']} (reader {nodule['reader']})")
    # Dibuja cada ROI del nódulo sobre su slice correspondiente
    for i, roi in enumerate(nodule["rois"], start=1):
        sl = find_slice(by_sop, by_z, roi)
        if sl is None:
            print(f"[WARN] No se encontró slice para ROI #{i}")
            continue
        title = f"ROI #{i} – Z={sl['z']:.3f} – {os.path.basename(sl['path'])}"
        plot_roi_on_slice(sl["arr"], roi["edge"], title=title)


SystemExit: No se encontraron nódulos en el XML.

c:\Users\Diego Castañeda\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
